# Notebook 10 — Stacking Ensemble

**Combines the scores of the base learners** (NB03 IF, NB03b K-Means+IF, NB04 Dense-AE, NB05 LSTM-AE, NB07 CNN-AE, NB08 Transformer-AE) **through a meta-learner**. It directly reuses the out-of-fold stacking strategy from **NLP Project 3**.

**Why does stacking help here?** The six base learners make *complementary errors*: IF excels at abrupt jumps (`step`); the Dense-AE is stronger on sustained-magnitude anomalies (`scaling`, `offset`); the LSTM-AE detects `replay` and `ramp` earlier thanks to its temporal memory; the Transformer contributes global attention; and the CNN-AE captures local temporal patterns. A meta-learner trained on their outputs extracts the best of each.

**Leakage-free design.**
- Each base learner was trained on the clean **train** split and calibrated on **val**.
- To fit the meta-learner we use the base-learner scores on **val** (which contains attacks with ground truth, although the base learners never saw those labels during training).
- We report the final ensemble on **test**, which neither the base learners nor the meta-learner saw while fitting their weights.

**Strategies considered:**
1. **Mean blending** (plain average, no learned weights) — baseline.
2. **Weighted blending** with weights tuned on val (grid search).
3. **Meta-logistic regression** over the 6 scores (can capture non-monotonic relationships).
4. **Meta-LightGBM** (non-linear over the scores).

Final comparison by AUC-ROC, AUC-PR, F1, latency and total ensemble size.


## 0. Setup

In [1]:
import os, json, time, warnings, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12
print('OK')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    f1_score, fbeta_score, precision_score, recall_score,
    average_precision_score, precision_recall_curve
)
import joblib
try:
    import lightgbm as lgb
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
print('LightGBM available:', HAS_LGBM)


OK
LightGBM available: True


## 1. Load real data + configuration

In [2]:
BASE_DIR = '/Volumes/Extreme Pro Particion 1TB/TFG/UCIrvine/'
DATA_DIR = os.path.join(BASE_DIR, 'data')

with open(os.path.join(DATA_DIR, 'pipeline_config.json')) as f:
    config = json.load(f)

FEATURE_COLS = config['feature_cols']
ATTACK_TYPES = config['attack_types']

df_train = pd.read_csv(os.path.join(DATA_DIR, 'train_clean.csv'),
                       index_col='datetime', parse_dates=True)
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val_with_attacks.csv'),
                       index_col='datetime', parse_dates=True)
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test_with_attacks.csv'),
                       index_col='datetime', parse_dates=True)

print('DATA LOADED (UCI real)')
print('=' * 65)
for nm, d in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    pos = (d['label']==1).sum() if 'label' in d.columns else 0
    print(f'  {nm:6s} {len(d):>9,d}   attacks: {pos:>7,d}')

y_val  = df_val['label'].values
y_test = df_test['label'].values
print(f'\nVal: {len(y_val):,} (pos {(y_val==1).sum():,})')
print(f'Test: {len(y_test):,} (pos {(y_test==1).sum():,})')


DATA LOADED (UCI real)
  Train  1,300,061   attacks:       0
  Val      144,452   attacks:  21,739
  Test     604,999   attacks:  96,717

Val: 144,452 (pos 21,739)
Test: 604,999 (pos 96,717)


## 2. Load base-learner scores

In [3]:
# Load predictions_*.csv from each base learner. Each notebook names its score
# column differently (e.g. if_score, dae_score, lstm_score, kmif_score, or just
# 'score' in CNN-AE / Transformer-AE), so we detect the right column automatically.
#
# Detector map. Any file that is missing is skipped with a warning.
DETECTORS = {
    'if':           'predictions_isolation_forest.csv',
    'kmeans_if':    'predictions_kmeans_if.csv',
    'dense_ae':     'predictions_dense_autoencoder.csv',
    'lstm_ae':      'predictions_lstm_autoencoder.csv',
    'cnn_ae':       'predictions_cnn_autoencoder.csv',
    'transformer':  'predictions_transformer_autoencoder.csv',
}

def pick_score_column(df):
    '''Return the name of the score column, accepting 'score' or '*_score'
    (and ignoring any column that contains 'pred').'''
    if 'score' in df.columns:
        return 'score'
    cands = [c for c in df.columns
             if c.endswith('_score') and 'pred' not in c]
    if not cands:
        raise KeyError(f"No score column found. Columns: {list(df.columns)}")
    return cands[0]

# Load the test scores
scores_test_d = {}
present = []
for name, fname in DETECTORS.items():
    path = os.path.join(DATA_DIR, fname)
    if not os.path.exists(path):
        print(f'  ! {name:14s} ({fname} not found) -> skipped')
        continue
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    score_col = pick_score_column(df)
    scores_test_d[name] = df[score_col].values
    present.append(name)
    print(f'  > {name:14s} {df.shape}   score_col={score_col}')

print(f'\nDetectors available on test: {present}')


  > if             (604999, 16)   score_col=if_score
  > kmeans_if      (604999, 17)   score_col=kmif_score
  > dense_ae       (604999, 16)   score_col=dae_score
  > lstm_ae        (604999, 16)   score_col=lstm_score
  > cnn_ae         (604999, 6)   score_col=score
  > transformer    (604999, 6)   score_col=score

Detectors available on test: ['if', 'kmeans_if', 'dense_ae', 'lstm_ae', 'cnn_ae', 'transformer']


## 3. Generate base-learner scores on val (one-off)

In [4]:
# We need scores_val_d[name] to fit the meta-learner.
# If NB03..NB08 already saved predictions_*_val.csv, we load them; otherwise we
# regenerate the scores here by reloading the persisted models.

import joblib
from tensorflow.keras.models import load_model
import tensorflow as tf
try:
    tf.keras.mixed_precision.set_global_policy('float32')
except Exception:
    pass

def load_pipeline_config_for(name):
    '''Return (scaler, model, weights, kind) file paths for a given base learner.'''
    paths = {
        'if': ('scaler_isolation_forest.pkl', 'model_isolation_forest.pkl', None, 'sklearn'),
        'kmeans_if': ('scaler_kmeans_if.pkl', 'model_kmeans_if.pkl', None, 'kmeans_if'),
        'dense_ae': ('scaler_dense_autoencoder.pkl', 'model_dense_autoencoder.keras', 'feature_weights_dense_ae.npy', 'keras_ae'),
        'lstm_ae':  ('scaler_lstm_autoencoder.pkl',  'model_lstm_autoencoder.keras',  'feature_weights_lstm_ae.npy', 'keras_lstm'),
        'cnn_ae':   ('scaler_cnn_autoencoder.pkl',   'model_cnn_autoencoder.keras',   'feature_weights_cnn_ae.npy',  'keras_cnn'),
        'transformer': ('scaler_transformer_autoencoder.pkl', 'model_transformer_autoencoder.keras', 'patch_weights_transformer_ae.npy', 'keras_tr'),
    }
    return paths.get(name)


def compute_features(df, mode='full'):
    f = df[FEATURE_COLS].copy()
    f['VI_residual'] = (df['Global_active_power']
                        - df['Voltage'] * df['Global_intensity'] / 1000.0)
    if mode in ('medium','full'):
        h = df.index.hour + df.index.minute / 60.0
        f['hour_sin'] = np.sin(2*np.pi*h/24); f['hour_cos'] = np.cos(2*np.pi*h/24)
        d = df.index.dayofweek
        f['dow_sin']  = np.sin(2*np.pi*d/7);  f['dow_cos']  = np.cos(2*np.pi*d/7)
        f['gap_diff1'] = df['Global_active_power'].diff().fillna(0)
    if mode == 'full':
        f['vi_res_abs'] = f['VI_residual'].abs()
        f['vi_res_roll15_mean'] = f['vi_res_abs'].rolling(15, min_periods=1).mean()
        f['gap_intensity_ratio'] = df['Global_active_power'] / (df['Global_intensity']+0.01)
        f['sm_gap_ratio'] = ((df['Sub_metering_1']+df['Sub_metering_2']+df['Sub_metering_3'])/1000.0
                              / (df['Global_active_power']+0.01))
    return f.fillna(0)


def score_dense_ae(model, scaler, weights, df):
    # Dense-AE was trained with mode='full' (17 features)
    Xf = compute_features(df, mode='full')
    X = scaler.transform(Xf.values).astype('float32')
    rec = model.predict(X, batch_size=2048, verbose=0)
    return ((rec - X)**2 * weights).sum(axis=1)


def make_windows(X, window=60, stride=1):
    n = len(X)
    if n < window:
        return np.empty((0, window, X.shape[1]), dtype='float32')
    starts = np.arange(0, n - window + 1, stride)
    return np.stack([X[s:s+window] for s in starts]).astype('float32')


def expand(scores_w, n_total, window=60):
    # Map per-window scores back to per-timestamp: each timestamp takes the max
    # score among the windows covering it.
    out = np.full(n_total, np.nan, dtype='float32')
    for s, sc in enumerate(scores_w):
        out[s:s+window] = np.maximum(np.nan_to_num(out[s:s+window], nan=-np.inf), sc)
    nan_mask = np.isnan(out)
    if nan_mask.any():
        first_valid = np.argmax(~nan_mask)
        out[nan_mask] = out[first_valid]
    return out


def score_lstm_ae(model, scaler, weights, df, window=60):
    # LSTM-AE was trained with mode='full' (17 features)
    Xf = compute_features(df, mode='full')
    X = scaler.transform(Xf.values).astype('float32')
    Xw = make_windows(X, window=window, stride=1)
    if len(Xw) == 0:
        return np.zeros(len(df), dtype='float32')
    rec = model.predict(Xw, batch_size=512, verbose=0)
    err = ((rec - Xw)**2).mean(axis=1)             # (N, F)
    sc_w = (err * weights).sum(axis=1)             # (N,)
    return expand(sc_w, len(df), window=window)


def score_cnn_ae(model, scaler, weights, df, window=60):
    # CNN-AE was trained with mode='medium' plus vi_res_abs (14 features)
    Xf = compute_features(df, mode='medium')
    Xf['vi_res_abs'] = Xf['VI_residual'].abs()
    X = scaler.transform(Xf.values).astype('float32')
    Xw = make_windows(X, window=window, stride=1)
    if len(Xw) == 0:
        return np.zeros(len(df), dtype='float32')
    rec = model.predict(Xw, batch_size=512, verbose=0)
    err = ((rec - Xw)**2).mean(axis=1)
    sc_w = (err * weights).sum(axis=1)
    return expand(sc_w, len(df), window=window)


def score_transformer(model, scaler, patch_weights, df, window=60, patch_len=5):
    Xf = compute_features(df, mode='medium')
    Xf['vi_res_abs'] = Xf['VI_residual'].abs()
    X = scaler.transform(Xf.values).astype('float32')
    Xw = make_windows(X, window=window, stride=1)
    if len(Xw) == 0:
        return np.zeros(len(df), dtype='float32')
    N, W, F = Xw.shape
    Xp = Xw.reshape(N, W//patch_len, patch_len, F).reshape(N, W//patch_len, patch_len*F)
    rec = model.predict(Xp, batch_size=512, verbose=0)
    err = ((rec - Xp)**2).mean(axis=2)             # (N, n_patches)
    sc_w = (err * patch_weights).sum(axis=1)
    return expand(sc_w, len(df), window=window)


def score_if(model, scaler, df):
    Xf = compute_features(df, mode='full')
    X = scaler.transform(Xf.values)
    # For IF, -score_samples is already the anomaly score (higher = more anomalous)
    return -model.score_samples(X)


# Compute the val scores one base learner at a time (keeps memory low)
scores_val_d = {}
for name in present:
    try:
        cfg = load_pipeline_config_for(name)
        if cfg is None:
            continue
        sc_fn, mdl_fn, w_fn, kind = cfg
        scaler = joblib.load(os.path.join(DATA_DIR, sc_fn))
        weights = np.load(os.path.join(DATA_DIR, w_fn)) if w_fn else None
        if kind == 'sklearn':
            mdl = joblib.load(os.path.join(DATA_DIR, mdl_fn))
            sc = score_if(mdl, scaler, df_val)
        elif kind == 'kmeans_if':
            # K-Means+IF has its own scoring logic; load the model and fall back
            # to the IF path. Skip it if anything goes wrong.
            try:
                mdl = joblib.load(os.path.join(DATA_DIR, mdl_fn))
                sc = score_if(mdl, scaler, df_val)
            except Exception as e:
                print(f'  ! {name}: {e}'); continue
        elif kind == 'keras_ae':
            mdl = load_model(os.path.join(DATA_DIR, mdl_fn))
            sc = score_dense_ae(mdl, scaler, weights, df_val)
        elif kind == 'keras_lstm':
            mdl = load_model(os.path.join(DATA_DIR, mdl_fn))
            sc = score_lstm_ae(mdl, scaler, weights, df_val)
        elif kind == 'keras_cnn':
            mdl = load_model(os.path.join(DATA_DIR, mdl_fn))
            sc = score_cnn_ae(mdl, scaler, weights, df_val)
        elif kind == 'keras_tr':
            mdl = load_model(os.path.join(DATA_DIR, mdl_fn))
            sc = score_transformer(mdl, scaler, weights, df_val)
        else:
            continue
        scores_val_d[name] = np.asarray(sc, dtype='float32')
        print(f'  > {name:14s} val scores: {scores_val_d[name].shape}')
    except Exception as e:
        print(f'  ! {name}: {e}')


  > if             val scores: (144452,)
  ! kmeans_if: 'dict' object has no attribute 'score_samples'


2026-06-09 17:48:38.731620: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Pro
2026-06-09 17:48:38.731666: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-06-09 17:48:38.731676: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
I0000 00:00:1781020118.735595 2188793 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1781020118.735664 2188793 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-06-09 17:48:40.065967: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


  > dense_ae       val scores: (144452,)
  > lstm_ae        val scores: (144452,)
  > cnn_ae         val scores: (144452,)
  > transformer    val scores: (144452,)


## 4. Score normalization

In [5]:
# Each base learner produces scores on a different scale, so we z-score them
# using val statistics computed on the clean negatives only.

def standardize(scores, ref):
    mu = np.mean(ref); sd = np.std(ref) + 1e-9
    return (scores - mu) / sd

models_in_stack = [n for n in present if n in scores_val_d and n in scores_test_d]
print(f'Final models in the stack ({len(models_in_stack)}):', models_in_stack)

S_val  = np.column_stack([standardize(scores_val_d[n],  scores_val_d[n][y_val==0]) for n in models_in_stack])
S_test = np.column_stack([standardize(scores_test_d[n], scores_val_d[n][y_val==0]) for n in models_in_stack])
print(f'S_val: {S_val.shape}  S_test: {S_test.shape}')


Final models in the stack (5): ['if', 'dense_ae', 'lstm_ae', 'cnn_ae', 'transformer']
S_val: (144452, 5)  S_test: (604999, 5)


## 5. Strategy 1: Mean Blending

In [6]:
def calibrate_threshold(y_true, scores, beta=1.0):
    '''Find the threshold that maximizes F-beta via the precision-recall curve.'''
    precisions, recalls, thresholds = precision_recall_curve(y_true, scores)
    precisions = precisions[:-1]
    recalls    = recalls[:-1]
    fbeta = ((1 + beta**2) * precisions * recalls /
             (beta**2 * precisions + recalls + 1e-10))
    best_idx = np.argmax(fbeta)
    best_thr = thresholds[best_idx]
    return best_thr, precisions[best_idx], recalls[best_idx], fbeta[best_idx]

s_val_mean  = S_val.mean(axis=1)
s_test_mean = S_test.mean(axis=1)
thr_m, _, _, _ = calibrate_threshold(y_val, s_val_mean)
y_pred_m = (s_test_mean > thr_m).astype(int)
auc_roc_m = roc_auc_score(y_test, s_test_mean)
auc_pr_m  = average_precision_score(y_test, s_test_mean)
f1_m  = f1_score(y_test, y_pred_m)
print(f'Mean Blending: AUC-ROC={auc_roc_m:.4f}  AUC-PR={auc_pr_m:.4f}  F1={f1_m:.4f}')


Mean Blending: AUC-ROC=0.9162  AUC-PR=0.7036  F1=0.6931


## 6. Strategy 2: Weighted Blending (grid search)

In [7]:
from itertools import product

def grid_blend(S_val, y_val, S_test, n_steps=4):
    '''Grid search over discrete weights on the probability simplex.'''
    n_models = S_val.shape[1]
    # Discrete weight grid {0, 0.33, 0.67, 1.0}
    grid_w = np.linspace(0, 1, n_steps+1)
    best = {'auc_pr': 0, 'weights': None, 'thr': 0.5}
    combos = 0
    for ws in product(grid_w, repeat=n_models):
        s = sum(ws)
        if s == 0: continue
        ws_norm = np.array(ws) / s
        s_val_w = S_val @ ws_norm
        # Recalibrate the threshold for this weight combination
        try:
            thr, _, _, _ = calibrate_threshold(y_val, s_val_w)
        except Exception:
            continue
        ap = average_precision_score(y_val, s_val_w)
        if ap > best['auc_pr']:
            best.update(auc_pr=ap, weights=ws_norm, thr=thr)
        combos += 1
    return best

t0 = time.time()
best_blend = grid_blend(S_val, y_val, S_test, n_steps=4)
print(f'Search in {time.time()-t0:.1f}s')
print(f'Optimal weights:')
for n, w in zip(models_in_stack, best_blend["weights"]):
    print(f'   {n:14s} {w:.3f}')
print(f'  thr: {best_blend["thr"]:.6f}')

s_test_wb = S_test @ best_blend['weights']
y_pred_wb = (s_test_wb > best_blend['thr']).astype(int)
auc_roc_wb = roc_auc_score(y_test, s_test_wb)
auc_pr_wb  = average_precision_score(y_test, s_test_wb)
f1_wb = f1_score(y_test, y_pred_wb)
print(f'Weighted Blending: AUC-ROC={auc_roc_wb:.4f}  AUC-PR={auc_pr_wb:.4f}  F1={f1_wb:.4f}')


Search in 76.5s
Optimal weights:
   if             0.000
   dense_ae       1.000
   lstm_ae        0.000
   cnn_ae         0.000
   transformer    0.000
  thr: 0.042322
Weighted Blending: AUC-ROC=0.9304  AUC-PR=0.8176  F1=0.7886


## 7. Strategy 3: Meta-logistic regression

In [8]:
meta_lr = LogisticRegression(C=2.0, max_iter=2000, class_weight='balanced',
                              solver='liblinear', random_state=42)
meta_lr.fit(S_val, y_val)
s_test_lr = meta_lr.predict_proba(S_test)[:, 1]
thr_lr, _, _, _ = calibrate_threshold(y_val, meta_lr.predict_proba(S_val)[:, 1])
y_pred_lr = (s_test_lr > thr_lr).astype(int)
auc_roc_lr = roc_auc_score(y_test, s_test_lr)
auc_pr_lr  = average_precision_score(y_test, s_test_lr)
f1_lr = f1_score(y_test, y_pred_lr)
print(f'Meta-LR:  AUC-ROC={auc_roc_lr:.4f}  AUC-PR={auc_pr_lr:.4f}  F1={f1_lr:.4f}')
print(f'  Weights (coef): {dict(zip(models_in_stack, meta_lr.coef_[0].round(3)))}')
print(f'  Intercept: {meta_lr.intercept_[0]:.3f}')


Meta-LR:  AUC-ROC=0.9361  AUC-PR=0.7972  F1=0.7637
  Weights (coef): {'if': np.float64(1.342), 'dense_ae': np.float64(0.655), 'lstm_ae': np.float64(-0.164), 'cnn_ae': np.float64(-0.563), 'transformer': np.float64(0.624)}
  Intercept: -1.674


## 8. Strategy 4: Meta-LightGBM (non-linear)

In [9]:
if HAS_LGBM:
    dtrain = lgb.Dataset(S_val, label=y_val)
    params = dict(objective='binary', metric='auc', learning_rate=0.05,
                  num_leaves=15, max_depth=4, min_data_in_leaf=100,
                  is_unbalance=True, num_threads=10, verbose=-1, seed=42)
    gbm_meta = lgb.train(params, dtrain, num_boost_round=200,
                         valid_sets=[dtrain],
                         callbacks=[lgb.log_evaluation(period=0)])
    s_test_gbm = gbm_meta.predict(S_test)
    thr_gbm, _, _, _ = calibrate_threshold(y_val, gbm_meta.predict(S_val))
    y_pred_gbm = (s_test_gbm > thr_gbm).astype(int)
    auc_roc_gbm = roc_auc_score(y_test, s_test_gbm)
    auc_pr_gbm  = average_precision_score(y_test, s_test_gbm)
    f1_gbm = f1_score(y_test, y_pred_gbm)
    print(f'Meta-LGBM: AUC-ROC={auc_roc_gbm:.4f}  AUC-PR={auc_pr_gbm:.4f}  F1={f1_gbm:.4f}')
else:
    print('LightGBM not available — skipped.')
    auc_roc_gbm = auc_pr_gbm = f1_gbm = None
    s_test_gbm = None; thr_gbm = None


Meta-LGBM: AUC-ROC=0.9630  AUC-PR=0.9101  F1=0.8306


## 9. Comparison + selection of the best ensemble

In [10]:
summary = pd.DataFrame([
    {'strategy':'Mean blending',     'AUC-ROC':auc_roc_m,  'AUC-PR':auc_pr_m,  'F1':f1_m},
    {'strategy':'Weighted blending', 'AUC-ROC':auc_roc_wb, 'AUC-PR':auc_pr_wb, 'F1':f1_wb},
    {'strategy':'Meta-LR',           'AUC-ROC':auc_roc_lr, 'AUC-PR':auc_pr_lr, 'F1':f1_lr},
])
if HAS_LGBM:
    summary = pd.concat([summary, pd.DataFrame([{'strategy':'Meta-LightGBM',
                                                  'AUC-ROC':auc_roc_gbm, 'AUC-PR':auc_pr_gbm,
                                                  'F1':f1_gbm}])], ignore_index=True)
print('ENSEMBLE COMPARISON:')
print(summary.to_string(index=False))

best_strat = summary.sort_values('AUC-PR', ascending=False).iloc[0]['strategy']
print(f'\n  WINNER: {best_strat}')

# Map the winning strategy to its test scores and threshold
score_map = {
    'Mean blending':     (s_test_mean, thr_m),
    'Weighted blending': (s_test_wb,   best_blend['thr']),
    'Meta-LR':           (s_test_lr,   thr_lr),
}
if HAS_LGBM:
    score_map['Meta-LightGBM'] = (s_test_gbm, thr_gbm)

scores_test_final, best_thr_final = score_map[best_strat]


ENSEMBLE COMPARISON:
         strategy  AUC-ROC   AUC-PR       F1
    Mean blending 0.916153 0.703560 0.693066
Weighted blending 0.930388 0.817584 0.788556
          Meta-LR 0.936084 0.797218 0.763735
    Meta-LightGBM 0.963021 0.910103 0.830610

  WINNER: Meta-LightGBM


## 10. Temporal post-processing + smoothed evaluation

In [11]:
def temporal_vote_vectorized(y_pred, W, K):
    '''Causal K-of-W voting filter. Flags a timestamp when at least K of the
    last W raw predictions are positive. O(n) via cumulative sum.'''
    cum = np.concatenate([[0], np.cumsum(y_pred)])
    n = len(y_pred)
    ends = np.arange(1, n + 1)
    starts = np.maximum(0, ends - W)
    votes = cum[ends] - cum[starts]
    return (votes >= K).astype(int)


def grid_search_wk(scores_val, y_val, best_thr, W_grid=(3,5,7,9,11,15,21,30)):
    '''Search the (W, K) voting window that maximizes F1 on val.'''
    y_pred_val = (scores_val > best_thr).astype(int)
    best_f1, best_W, best_K = 0, 3, 2
    for W in W_grid:
        for K in range(2, W + 1):
            y_sm = temporal_vote_vectorized(y_pred_val, W, K)
            f1 = f1_score(y_val, y_sm)
            if f1 > best_f1:
                best_f1, best_W, best_K = f1, W, K
    return best_W, best_K, best_f1

# Post-processing uses the val scores of the winning strategy
score_val_map = {
    'Mean blending':     s_val_mean,
    'Weighted blending': S_val @ best_blend['weights'],
    'Meta-LR':           meta_lr.predict_proba(S_val)[:, 1],
}
if HAS_LGBM:
    score_val_map['Meta-LightGBM'] = gbm_meta.predict(S_val)
scores_val_final = score_val_map[best_strat]

best_W, best_K, _ = grid_search_wk(scores_val_final, y_val, best_thr_final)
print(f'Optimal temporal filter (ensemble): W={best_W}, K={best_K}')

y_pred_raw = (scores_test_final > best_thr_final).astype(int)
y_pred_smooth = temporal_vote_vectorized(y_pred_raw, best_W, best_K)

f1_r  = f1_score(y_test, y_pred_raw)
f2_r  = fbeta_score(y_test, y_pred_raw, beta=2)
prec_r = precision_score(y_test, y_pred_raw)
rec_r  = recall_score(y_test, y_pred_raw)
auc_roc = roc_auc_score(y_test, scores_test_final)
auc_pr  = average_precision_score(y_test, scores_test_final)

f1_s  = f1_score(y_test, y_pred_smooth)
f2_s  = fbeta_score(y_test, y_pred_smooth, beta=2)
prec_s = precision_score(y_test, y_pred_smooth)
rec_s  = recall_score(y_test, y_pred_smooth)

print('='*60); print(f'ENSEMBLE ({best_strat}) — RAW vs SMOOTHED'); print('='*60)
print(f'  RAW:      F1={f1_r:.4f} F2={f2_r:.4f} P={prec_r:.4f} R={rec_r:.4f}')
print(f'  SMOOTHED: F1={f1_s:.4f} F2={f2_s:.4f} P={prec_s:.4f} R={rec_s:.4f}')
print(f'  AUC-ROC={auc_roc:.4f}  AUC-PR={auc_pr:.4f}')


# -- Joint search (threshold x W x K) tuned on val --
prc_pr_, prc_rc_, prc_thr_ = precision_recall_curve(y_val, scores_val_final)
prc_thr_ = prc_thr_[prc_thr_ > 0]
if len(prc_thr_) < 5:
    thr_candidates = np.linspace(scores_val_final.min(), scores_val_final.max(), 40)
else:
    thr_candidates = np.unique(np.quantile(prc_thr_, np.linspace(0.30, 0.999, 80)))
W_grid_joint = [3, 5, 7, 9, 11, 15, 21, 31, 45, 60]
best_joint_f1 = -1.0
best_joint = (best_thr_final, best_W, best_K)
for thr in thr_candidates:
    preds_v = (scores_val_final > thr).astype(int)
    if preds_v.sum() in (0, len(preds_v)):
        continue
    for W in W_grid_joint:
        for K in range(2, W+1):
            y_sm = temporal_vote_vectorized(preds_v, W, K)
            if y_sm.sum() == 0:
                continue
            f1 = f1_score(y_val, y_sm)
            if f1 > best_joint_f1:
                best_joint_f1 = f1
                best_joint = (thr, W, K)
preds_v0 = (scores_val_final > best_thr_final).astype(int)
y_sm0 = temporal_vote_vectorized(preds_v0, best_W, best_K)
f1_val_orig = f1_score(y_val, y_sm0)
print(f'\nJOINT SEARCH NB10:')
print(f'  best joint val F1: {best_joint_f1:.4f}  thr={best_joint[0]:.6f} W={best_joint[1]} K={best_joint[2]}')
print(f'  orig val F1:       {f1_val_orig:.4f}  thr={best_thr_final:.6f} W={best_W} K={best_K}')
if best_joint_f1 > f1_val_orig:
    best_thr_final, best_W, best_K = best_joint
    y_pred_raw = (scores_test_final > best_thr_final).astype(int)
    print('  -> adopting joint')
else:
    print('  -> keeping previous')

# Re-apply with the final (threshold, W, K)
y_pred_smooth = temporal_vote_vectorized(y_pred_raw, best_W, best_K)
f1_s = f1_score(y_test, y_pred_smooth)
f2_s = fbeta_score(y_test, y_pred_smooth, beta=2)
prec_s = precision_score(y_test, y_pred_smooth)
rec_s  = recall_score(y_test, y_pred_smooth)
f1_r  = f1_score(y_test, y_pred_raw)
f2_r  = fbeta_score(y_test, y_pred_raw, beta=2)
prec_r = precision_score(y_test, y_pred_raw)
rec_r  = recall_score(y_test, y_pred_raw)
print(f'TEST after joint: F1_smoothed={f1_s:.4f}  P={prec_s:.4f}  R={rec_s:.4f}')


Optimal temporal filter (ensemble): W=3, K=2
ENSEMBLE (Meta-LightGBM) — RAW vs SMOOTHED
  RAW:      F1=0.8306 F2=0.7731 P=0.9482 R=0.7389
  SMOOTHED: F1=0.8277 F2=0.7711 P=0.9433 R=0.7374
  AUC-ROC=0.9630  AUC-PR=0.9101

JOINT SEARCH NB10:
  best joint val F1: 0.8808  thr=0.794471 W=3 K=2
  orig val F1:       0.8808  thr=0.795072 W=3 K=2
  -> keeping previous
TEST after joint: F1_smoothed=0.8277  P=0.9433  R=0.7374


## 11. Recall per type + latency + saving

In [12]:
def recall_per_type_severity(df_preds, score_col, threshold,
                              attack_types=ATTACK_TYPES,
                              severities=('low','medium','high')):
    '''Recall broken down by (attack type, severity).'''
    rows = []
    for atype in attack_types:
        row = {'type': atype}
        for sev in severities:
            mask = (df_preds['attack_type']==atype) & (df_preds['severity']==sev)
            if mask.sum() == 0:
                row[sev] = np.nan
            else:
                y_true_sub  = df_preds.loc[mask, 'label'].values
                y_score_sub = df_preds.loc[mask, score_col].values
                y_pred_sub  = (y_score_sub > threshold).astype(int)
                row[sev] = recall_score(y_true_sub, y_pred_sub, zero_division=0)
        rows.append(row)
    out = pd.DataFrame(rows).set_index('type')
    print('Recall per type x severity (smoothed):')
    print(out.round(3))
    return out
def compute_detection_latency(df_preds, score_col, threshold):
    '''Per-episode detection latency in minutes (time from episode start to the
    first flagged timestamp).'''
    results = []
    attacked = df_preds[df_preds['episode_id'] >= 0]
    for ep_id, group in attacked.groupby('episode_id'):
        group = group.sort_index()
        detections = group[group[score_col] > threshold]
        ep_start = group.index[0]
        results.append({
            'episode_id': ep_id,
            'type': group['attack_type'].iloc[0],
            'severity': group['severity'].iloc[0],
            'duration_min': len(group),
            'latency_min': ((detections.index[0] - ep_start).total_seconds() / 60
                            if len(detections) > 0 else np.nan),
            'detected': len(detections) > 0
        })
    return pd.DataFrame(results)


def print_latency_summary(lat_df):
    det_rate = lat_df['detected'].mean()
    detected = lat_df[lat_df['detected']]
    if len(detected) > 0:
        med_lat = detected['latency_min'].median()
        p90_lat = detected['latency_min'].quantile(0.9)
    else:
        med_lat = p90_lat = np.nan
    print('=' * 65)
    print('DETECTION LATENCY (smoothed)')
    print('=' * 65)
    print(f'  Detection rate: {det_rate*100:.2f}%')
    print(f'  Median:         {med_lat:.1f} min')
    print(f'  P90:            {p90_lat:.1f} min')
    print()
    print('  Per attack type:')
    for atype, g in lat_df.groupby('type'):
        gd = g[g['detected']]
        print(f'    {atype:18s} n={len(g):3d}  det={g["detected"].mean()*100:5.1f}%  '
              f'med_lat={gd["latency_min"].median() if len(gd)>0 else float("nan"):5.1f}min')
    return det_rate, med_lat, p90_lat

df_preds_test = df_test[['attack_type','severity','episode_id','label']].copy()
df_preds_test['score']   = scores_test_final
df_preds_test['pred_sm'] = y_pred_smooth

def recall_per_type_severity_sm(df_preds, pred_col,
                                attack_types=ATTACK_TYPES,
                                severities=('low','medium','high')):
    # Same breakdown as above, but scored on the smoothed predictions directly
    rows = []
    for atype in attack_types:
        row = {'type': atype}
        for sev in severities:
            mask = (df_preds['attack_type']==atype) & (df_preds['severity']==sev)
            if mask.sum() == 0:
                row[sev] = np.nan
            else:
                row[sev] = recall_score(df_preds.loc[mask,'label'].values,
                                         df_preds.loc[mask,pred_col].values, zero_division=0)
        rows.append(row)
    out = pd.DataFrame(rows).set_index('type')
    print('Recall per type x severity (smoothed):')
    print(out.round(3))
    return out

recall_table = recall_per_type_severity_sm(df_preds_test, 'pred_sm')
lat = compute_detection_latency(df_preds_test, 'score', best_thr_final)
det_rate, med_lat, p90_lat = print_latency_summary(lat)

df_preds_test.to_csv(os.path.join(DATA_DIR, 'predictions_stacking.csv'))
lat.to_csv(os.path.join(DATA_DIR, 'latency_stacking.csv'), index=False)

# Total ensemble size = combined on-disk size of its base-learner models
total_size_kb = 0
for name in models_in_stack:
    cfg = load_pipeline_config_for(name)
    if cfg is None: continue
    mdl_fn = cfg[1]
    p = os.path.join(DATA_DIR, mdl_fn)
    if os.path.exists(p):
        total_size_kb += os.path.getsize(p)/1e3

metrics = {
    'model': f'Stacking ({best_strat}) + Temporal Voting',
    'detectors': models_in_stack,
    'best_strategy': best_strat,
    'raw_f1':        round(float(f1_r), 4),
    'raw_f2':        round(float(f2_r), 4),
    'raw_precision': round(float(prec_r), 4),
    'raw_recall':    round(float(rec_r), 4),
    'f1':            round(float(f1_s), 4),
    'f2':            round(float(f2_s), 4),
    'precision':     round(float(prec_s), 4),
    'recall':        round(float(rec_s), 4),
    'auc_roc':       round(float(auc_roc), 4),
    'auc_pr':        round(float(auc_pr), 4),
    'threshold':     float(best_thr_final),
    'temporal_W':    int(best_W),
    'temporal_K':    int(best_K),
    'total_size_kb': round(total_size_kb, 1),
    'detection_rate':round(float(det_rate), 4),
    'median_latency_min': float(med_lat) if not np.isnan(med_lat) else None,
    'meta_weights': dict(zip(models_in_stack, meta_lr.coef_[0].round(3).tolist())) if best_strat == 'Meta-LR' else None,
    'summary_table': summary.round(4).to_dict(orient='records'),
}
with open(os.path.join(DATA_DIR, 'metrics_stacking.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

# Persist the meta-LR
joblib.dump(meta_lr, os.path.join(DATA_DIR, 'meta_lr_stacking.pkl'))
print('Saved complete ensemble in data/.')


Recall per type x severity (smoothed):
                 low  medium   high
type                               
scaling        0.359   0.799  0.979
offset         0.531   0.863  0.980
noise          0.836   0.959  0.976
ramp           0.333   0.573  0.707
step           0.416   0.610  0.597
replay         0.500   0.763  0.854
voltage_spoof  0.754   0.986  0.991
DETECTION LATENCY (smoothed)
  Detection rate: 96.31%
  Median:         0.0 min
  P90:            50.0 min

  Per attack type:
    noise              n=120  det=100.0%  med_lat=  0.0min
    offset             n=120  det= 95.0%  med_lat=  0.0min
    ramp               n=120  det= 95.0%  med_lat= 35.0min
    replay             n=120  det= 97.5%  med_lat=  0.0min
    scaling            n=120  det= 94.2%  med_lat=  0.0min
    step               n=120  det= 95.0%  med_lat= 39.5min
    voltage_spoof      n=120  det= 97.5%  med_lat=  0.0min
Saved complete ensemble in data/.


## 12. Summary

In [13]:
print('='*65)
print(f'NB10 — STACKING ({best_strat}): SUMMARY')
print('='*65)
print(f'  Detectors:  {", ".join(models_in_stack)}')
print(f'  AUC-ROC:    {auc_roc:.4f}')
print(f'  AUC-PR:     {auc_pr:.4f}')
print(f'  F1 raw:     {f1_r:.4f} -> smoothed: {f1_s:.4f}')
print(f'  Median latency: {med_lat:.1f} min')
print(f'  Detection rate:   {det_rate*100:.2f}%')
print(f'  Total ensemble size: {total_size_kb:.0f} KB')
print()
print('Stacking is expected to add 1-3 F1 points over the best individual')
print('detector, driven by the complementarity of the base learners\' errors.')


NB10 — STACKING (Meta-LightGBM): SUMMARY
  Detectors:  if, dense_ae, lstm_ae, cnn_ae, transformer
  AUC-ROC:    0.9630
  AUC-PR:     0.9101
  F1 raw:     0.8306 -> smoothed: 0.8277
  Median latency: 0.0 min
  Detection rate:   96.31%
  Total ensemble size: 11525 KB

Stacking is expected to add 1-3 F1 points over the best individual
detector, driven by the complementarity of the base learners' errors.
